# Training curve analysis

Compares runs by **group**, reading the per-epoch `epoch_metrics.csv` that
`scripts/train_gnn.py` writes to each run's `results/<run_name>/` directory, plus the
end-of-run `results/<run_name>.json` summary for the confusion matrices in section 5.

Five groups, because they answer different questions and sharing an axis makes each harder to
read:

1. **Aggregation at 10µm** — the ladder of how much learned mixing happens before the readout:
   mean pooling (none), MPNN (fixed local neighbour averaging over 2 hops), GraphTransformer
   (adjacency-biased global attention), all at the baseline's own 10µm window. The mean and
   MPNN runs are trained with `--spatial`, so all three see the same node inputs and the
   comparison isolates aggregation rather than confounding it with node features.
2–4. **Window radius**, one group per aggregator. Radius is the baseline's *own* aggregation
   parameter, so sweeping it asks how much of any gain is the aggregation method versus simply
   seeing more context. `0` is the identity case — a single node, no aggregation at all —
   `10µm` is the baseline's window and the reference point, and `20/40µm` widen it. Note that
   window count is one per node regardless of radius; only window *size* changes (mean 1 /
   10.7 / 22.1 / 49.2 nodes).
5. **Geometry only (40µm)** — `--no-embeddings` runs, where the SegCLR embedding is dropped
   from the node input and the model sees nothing but the graph, the center-relative offset and
   the Laplacian PE. Paired against the embedding-carrying run at the same radius: the gap
   between them is how much of the score is the embeddings and how much is the shape they sit
   on.

Section 2's **delta view** covers the groups listed in `DELTA_GROUPS` — those whose entries
differ from one shared reference by exactly one thing, where the signed change from that
reference is the meaningful quantity rather than the absolute bar height.

Classification runs at the **9-class level 2 of `hierarchy_v2`**
(`gnn/hierarchy.py::HIERARCHY_V2_TREE`, truncated by
`data/dataset_lcpn.py::HIERARCHY_LEVELS_DROPPED`), the taxonomy registered in the shared v3
store. Every per-class view below is therefore over pyramidal / thalamocortical / the three
interneuron families / the four glia — not the granular cell types. Support is steeply
imbalanced (pyramidal 1740 cells vs. OPC 3), so read balanced accuracy and per-class recall,
not raw accuracy; and OPC's *cell*-level recall can only ever read 0.0 or 1.0, since it has a
single test cell.

CSV columns: `epoch`, `train_loss`, `window_*` and `cell_*` accuracy / balanced_accuracy /
macro_precision / macro_f1, plus `window_recall_<class>` / `cell_recall_<class>` and the
matching `_precision_` columns per class. Per-class F1 is not logged — it's derived here from
precision + recall (`f1_from_pr`) rather than duplicated in the CSV.

Kernel: **segclr_db (.venv)** — needs pandas + matplotlib
(`scripts/sbatch/install_matplotlib.sh`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("..")
RESULTS_DIR = REPO_ROOT / "results"

# ---------------------------------------------------------------------------
# Configuration -- edit the groups below and re-run.
#
# Run names encode the aggregation method (scripts/train_gnn.py's agg_tag):
# "meanpool", "mpnn_L{layers}", or "gt_L{depth}_H{heads}", with any enabled
# switch appended -- _spatial for mean/mpnn, then _resnet{L}x{H}, then
# _w{radius}um for any non-default window radius, then _noemb for a
# geometry-only run. The 10um radius is deliberately untagged, since it is the
# baseline's own window and the reference point for the radius sweep.
# ---------------------------------------------------------------------------


def _csv(run: str):
    return RESULTS_DIR / f"gnn_lcpn_scratch_{run}" / "epoch_metrics.csv"


def _radius_csv(base: str, um: int):
    """Radius-tagged CSV path; 10um is the untagged default (see above)."""
    return _csv(base) if um == 10 else _csv(f"{base}_w{um}um")


# The aggregation ladder at the baseline's own 10um window: none, fixed local
# neighbour averaging, adjacency-biased global attention. All three see the
# same node inputs here -- mean and MPNN are run with --spatial so they get the
# Laplacian PE and center-relative offset the GT builds for itself, which is
# what makes this a comparison of aggregation rather than of node features.
ARCH_CSVS = [
    ("mean (raw)", _csv("meanpool")),
    ("MPNN L2 + spatial", _csv("mpnn_L2_spatial")),
    ("GT + spatial", _csv("gt_L4_H4")),
]

# Window radius sweep, one group per aggregator so the curves stay readable.
# Radius is the baseline's own aggregation parameter: 0 is the identity case
# (a single node, no aggregation at all), 10um is what the baseline uses, and
# 20/40um widen the geodesic context. Baseline FIRST in each list -- the delta
# view below takes the first entry as its reference.
MEAN_RADIUS_CSVS = [(f"mean {um}um", _radius_csv("meanpool", um)) for um in (10, 0, 20, 40)]
MPNN_RADIUS_CSVS = [(f"MPNN {um}um", _radius_csv("mpnn_L2_spatial", um)) for um in (10, 20, 40)]
GT_RADIUS_CSVS = [(f"GT {um}um", _radius_csv("gt_L4_H4", um)) for um in (10, 20, 40)]

# Geometry-only control (--no-embeddings): the SegCLR embedding is dropped
# from the node input, leaving the graph, the center-relative offset and the
# Laplacian PE. Paired with the embedding-carrying run at the same radius
# wherever that exists, because the whole point is the gap between them -- how
# much of a score is the embeddings and how much is the shape they sit on.
NOEMB_CSVS = [
    ("MPNN 40um + emb", _radius_csv("mpnn_L2_spatial", 40)),
    ("MPNN 40um geometry only", _csv("mpnn_L2_spatial_w40um_noemb")),
    ("GT 40um + emb", _radius_csv("gt_L4_H4", 40)),
    ("GT 40um geometry only", _csv("gt_L4_H4_w40um_noemb")),
]

COMPARISON_GROUPS = [
    ("Aggregation at 10um", ARCH_CSVS),
    ("Window radius: mean", MEAN_RADIUS_CSVS),
    ("Window radius: MPNN", MPNN_RADIUS_CSVS),
    ("Window radius: GT", GT_RADIUS_CSVS),
    ("Geometry only (40um)", NOEMB_CSVS),
]

# Groups whose entries differ from one shared reference by exactly one thing,
# so the signed delta from that reference is the meaningful quantity rather
# than the absolute bar height. Each pair is (group name, baseline label); the
# baseline must be the first entry of the corresponding list above.
DELTA_GROUPS = [
    ("Window radius: mean", "mean 10um"),
    ("Window radius: MPNN", "MPNN 10um"),
    ("Window radius: GT", "GT 10um"),
]

# Which metric selects each run's "best" epoch -- matches train_gnn.py's own
# checkpoint-selection criterion (best val_window_bacc: more stable than
# cell-level, which majority-votes only a few hundred val cells), so "best
# epoch" here means the same thing as the saved checkpoint_best.pt.
BEST_EPOCH_METRIC = "window_balanced_accuracy"

MANIFEST_PATH = REPO_ROOT / "data" / "manifest.json"  # for the class-support section

plt.rcParams["figure.dpi"] = 110

In [ ]:
def load_metrics(csv_path) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def class_names_from_columns(df: pd.DataFrame, prefix: str = "window_recall_") -> list[str]:
    return [c[len(prefix):] for c in df.columns if c.startswith(prefix)]


def best_epoch_row(df: pd.DataFrame, metric: str = BEST_EPOCH_METRIC) -> pd.Series:
    return df.loc[df[metric].idxmax()]


def f1_from_pr(precision, recall) -> np.ndarray:
    """Per-class F1 from precision + recall -- not logged to the CSV (only the
    macro F1 scalar is), so every plot that wants per-class F1 derives it here
    instead. Matches gnn/metrics.py::macro_f1's harmonic-mean formula, just
    element-wise over classes instead of pre-averaged."""
    p, r = np.asarray(precision, dtype=float), np.asarray(recall, dtype=float)
    denom = p + r
    return np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)


def has_precision_cols(df: pd.DataFrame) -> bool:
    return "window_macro_precision" in df.columns


def available_runs(pairs, group_name=""):
    """Drop entries whose CSV isn't on disk yet, with a note.

    Runs are submitted as a batch and land one at a time, so a group is
    routinely half-complete. Skipping with a printed note beats a
    FileNotFoundError that takes the whole notebook down mid-sweep.
    """
    have = [(label, path) for label, path in pairs if Path(path).exists()]
    missing = [label for label, path in pairs if not Path(path).exists()]
    if missing:
        print(f"  [{group_name}] not on disk yet, skipped: {', '.join(missing)}")
    return have


def grouped_bar(ax, x_labels, series_by_label, colors=None, ylabel="score", title="", ylim=(0, 1)):
    """series_by_label: {series_label: [value per x_label]}. One group of
    adjacent, differently-colored bars per x position."""
    n_series = len(series_by_label)
    x = np.arange(len(x_labels))
    width = 0.8 / max(n_series, 1)
    if colors is None:
        colors = plt.cm.tab10(np.linspace(0, 1, max(n_series, 2)))
    for i, (label, values) in enumerate(series_by_label.items()):
        offset = i * width - (n_series - 1) * width / 2
        ax.bar(x + offset, values, width, label=label, color=colors[i])
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=60 if len(x_labels) > 4 else 0,
                       ha="right" if len(x_labels) > 4 else "center")
    ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.legend(fontsize=8)


def load_group(pairs, group_name="", metric=BEST_EPOCH_METRIC) -> pd.DataFrame:
    """One row per run, taken at that run's own best epoch."""
    rows = []
    for label, path in available_runs(pairs, group_name):
        b = best_epoch_row(load_metrics(path), metric).copy()
        b["model"] = label
        rows.append(b)
    return pd.DataFrame(rows).reset_index(drop=True)

## 1. Headline comparison, per group

Each run evaluated at its **own** best epoch (by `BEST_EPOCH_METRIC`). Window-level metrics
average over ~2.4M test windows; cell-level metrics majority-vote those up to 467 cells and are
correspondingly noisier — which is exactly why checkpoint selection uses the window metric.

Window count is the same at every radius (one window per node), so the radius groups differ in
window *size*, not sample count.

In [ ]:
group_frames = {}
for group_name, pairs in COMPARISON_GROUPS:
    df_g = load_group(pairs, group_name)
    group_frames[group_name] = df_g
    if df_g.empty:
        print(f"### {group_name}: no runs on disk yet\n")
        continue
    cols = ["model", "epoch", "window_accuracy", "window_balanced_accuracy"]
    cols += [c for c in ("window_macro_precision", "window_macro_f1") if c in df_g.columns]
    cols += ["cell_accuracy", "cell_balanced_accuracy"]
    cols += [c for c in ("cell_macro_precision", "cell_macro_f1") if c in df_g.columns]
    print(f"### {group_name}")
    display(df_g[cols].round(4))

In [ ]:
for group_name, df_g in group_frames.items():
    if df_g.empty:
        continue
    metric_cols = ["window_accuracy", "window_balanced_accuracy"]
    metric_labels = ["window acc", "window balanced acc"]
    if "window_macro_precision" in df_g.columns and df_g["window_macro_precision"].notna().all():
        metric_cols += ["window_macro_precision", "window_macro_f1"]
        metric_labels += ["window precision", "window F1"]
    else:
        metric_cols += ["window_macro_f1"]
        metric_labels += ["window F1"]

    fig, ax = plt.subplots(figsize=(10, 5))
    series = {row["model"]: [row[c] for c in metric_cols] for _, row in df_g.iterrows()}
    grouped_bar(ax, metric_labels, series, title=f"{group_name} (each at its own best epoch)")
    plt.tight_layout()
    plt.show()

## 2. Per-class comparison, per group

Per-class window-level recall, precision and F1 at each run's best epoch. With support running
from 8.04M train windows (pyramidal) down to 2562 (OPC), the per-class view is where an
apparently healthy headline number turns out to rest on the populous classes alone.

For each group in `DELTA_GROUPS` a second view follows: the signed **delta from that group's
reference run**. Since those entries differ from the reference by exactly one thing — the
window radius — the delta is what that change actually did. Absolute bars mostly show the
shared behaviour of the underlying aggregator and bury the effect being measured.

In [ ]:
def per_class_series(pairs, group_name):
    """{label: [value per class]} for recall / precision / F1, plus the class list."""
    have = available_runs(pairs, group_name)
    if not have:
        return None, None, None, None
    classes = class_names_from_columns(load_metrics(have[0][1]))
    recall, precision, f1 = {}, {}, {}
    for label, path in have:
        d = load_metrics(path)
        b = best_epoch_row(d)
        recall[label] = [b[f"window_recall_{c}"] for c in classes]
        if has_precision_cols(d):
            precision[label] = [b[f"window_precision_{c}"] for c in classes]
            f1[label] = f1_from_pr(precision[label], recall[label])
    return classes, recall, precision, f1


for group_name, pairs in COMPARISON_GROUPS:
    classes, recall, precision, f1 = per_class_series(pairs, group_name)
    if not classes:
        continue
    w = max(10, len(classes) * 0.6)
    for series, ylabel in ((recall, "recall"), (precision, "precision"), (f1, "F1")):
        if not series or len(series) != len(recall):
            print(f"note: [{group_name}] a run predates per-class {ylabel} logging -- skipped.")
            continue
        fig, ax = plt.subplots(figsize=(w, 5))
        grouped_bar(ax, classes, series, ylabel=f"window-level {ylabel}",
                    title=f"{group_name}: per-class window {ylabel}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Delta view -- for groups whose entries differ from one reference by exactly
# one thing (DELTA_GROUPS). Absolute per-class bars are dominated by the shared
# behaviour of the underlying model and bury the effect being measured; the
# signed difference is the effect.
_pairs_by_name = dict(COMPARISON_GROUPS)

for group_name, baseline_label in DELTA_GROUPS:
    pairs = _pairs_by_name.get(group_name)
    if pairs is None:
        print(f"delta view: no group named {group_name!r} -- skipped.")
        continue
    classes, recall, precision, f1 = per_class_series(pairs, group_name)
    if not classes or baseline_label not in recall or len(recall) < 2:
        print(f"delta view [{group_name}]: needs {baseline_label!r} plus at least one "
              "other run on disk -- skipped.")
        continue
    for series, ylabel in ((recall, "recall"), (f1, "F1")):
        if not series or baseline_label not in series:
            continue
        base = np.asarray(series[baseline_label], dtype=float)
        deltas = {
            label: np.asarray(vals, dtype=float) - base
            for label, vals in series.items()
            if label != baseline_label
        }
        if not deltas:
            continue
        lim = float(np.abs(np.concatenate(list(deltas.values()))).max()) * 1.15 or 0.01
        fig, ax = plt.subplots(figsize=(max(10, len(classes) * 0.6), 5))
        grouped_bar(ax, classes, deltas, ylabel=f"delta window {ylabel} vs baseline",
                    title=f"{group_name}: change in per-class {ylabel} vs {baseline_label}",
                    ylim=(-lim, lim))
        ax.axhline(0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()

## 3. Training curves, per group

Train loss and validation window balanced accuracy against epoch, one line per run. This is
where convergence and overfitting show up: a run whose train loss keeps falling while its
validation metric flattens has stopped generalizing, and a run still climbing at the last
epoch was cut short rather than converged.

Note "val" is an alias for the test split (see `data/build_dataset_from_store.py`), so these
curves are not held out from the reported numbers — an accepted trade-off of the two-way
split.

In [ ]:
for group_name, pairs in COMPARISON_GROUPS:
    have = available_runs(pairs, group_name)
    if not have:
        continue
    fig, (ax_loss, ax_bacc) = plt.subplots(1, 2, figsize=(13, 4.5))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(have), 2)))
    for (label, path), color in zip(have, colors):
        d = load_metrics(path)
        ax_loss.plot(d["epoch"], d["train_loss"], label=label, color=color)
        ax_bacc.plot(d["epoch"], d["window_balanced_accuracy"], label=label, color=color)
    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("train loss")
    ax_loss.set_title(f"{group_name}: train loss")
    ax_bacc.set_xlabel("epoch"); ax_bacc.set_ylabel("val window balanced accuracy")
    ax_bacc.set_title(f"{group_name}: val window balanced accuracy")
    for a in (ax_loss, ax_bacc):
        a.legend(fontsize=8)
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Class support (train-split window counts)

Context for the per-class bars above — a class with very few training windows can swing wildly
on recall from run to run purely from noise, which the recall number alone doesn't show. Reads
`data/manifest.json` directly (same computation as
`data/dataset_lcpn.py::train_window_counts_by_label`).

In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT.resolve()))
from data.dataset_lcpn import load_hierarchy  # noqa: E402


def train_window_counts_by_class(manifest_path) -> dict[str, int]:
    """Train-split window count per CLASSIFIED class.

    manifest.json stores each cell's granular cell_type, but classification
    runs at a coarser level, so the granular counts are folded up through the
    same hierarchy the training code uses -- keying the raw cell_type against
    the coarse class names would match nothing and silently report zero
    support everywhere.

    load_hierarchy() is the single source of that mapping: it resolves the
    active tree AND applies HIERARCHY_LEVELS_DROPPED, so this cannot drift
    from what the runs were actually trained against the way a locally
    re-derived hierarchy would.
    """
    manifest = json.loads(Path(manifest_path).read_text())
    hierarchy = load_hierarchy(manifest)
    counts: dict[str, int] = {}
    for info in manifest["cells"].values():
        if info["split"] != "train":
            continue
        path = hierarchy.label_paths.get(info["cell_type"])
        if path is None:
            continue
        counts[path[-1]] = counts.get(path[-1], 0) + info["n_nodes_covered"]
    return counts


_all_pairs = [pair for _, pairs in COMPARISON_GROUPS for pair in pairs]
_first = next((p for _, p in _all_pairs if Path(p).exists()), None)
if _first is None:
    print("no runs on disk yet -- class list comes from a run's CSV columns.")
else:
    classes = class_names_from_columns(load_metrics(_first))
    support = train_window_counts_by_class(MANIFEST_PATH)
    missing = [c for c in classes if c not in support]
    if missing:
        print(f"note: no train support for {missing} -- these cannot be learned.")
    fig, ax = plt.subplots(figsize=(max(8, len(classes) * 0.5), 4))
    ax.bar(classes, [support.get(c, 0) for c in classes], color="#55A868")
    ax.set_yscale("log")
    ax.set_ylabel("train window count (log scale)")
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=60, ha="right")
    ax.set_title("Class support (train split)")
    plt.tight_layout()
    plt.show()

## 5. Confusion matrices

One heatmap per run, gridded per comparison group, so a whole sweep reads at a glance.

These come from `results/<run>.json`, not `epoch_metrics.csv`. The CSV logs only scalars and
per-class recall/precision, and a confusion matrix cannot be reconstructed from those — recall
fixes the row sums and precision the column sums, but neither pins down the off-diagonal mass.
The JSON matrices are the **best-epoch checkpoint's** predictions on the test split:
`scripts/train_gnn.py` reloads `checkpoint_best.pt` before its final evaluation, so these agree
with the best-epoch rows in section 1 rather than with the last epoch.

Rows are normalized to sum to 1, which makes the diagonal exactly per-class recall and lets
every panel share one 0–1 colour scale. That sharing is the point of a grid: raw counts are not
comparable across panels, because the largest class holds most of the mass and would saturate
its own row in every model. The number under each title is balanced accuracy recomputed from
the matrix (mean diagonal over classes that have support), as a check that the panel and
section 1 agree.

Off-diagonal structure is what a scalar cannot show. Two models with identical balanced
accuracy can fail in completely different ways — one spreading errors evenly, another
collapsing a whole family into its most populous sibling — and with an LCPN that difference is
usually the interesting part of an ablation, since a top-down cascade can only recover from a
mistake made at a coarser level if the coarser level got it right.

Both granularities are plotted: **cell-level** (majority-voted per `root_id`, the headline
number) and **window-level** (per-window, the diagnostic).

In [ ]:
# Cell-level ("test_metrics") is the headline; window-level is the diagnostic.
CM_GRANULARITIES = [("cell", "test_metrics"), ("window", "window_test_metrics")]

# Annotating every cell is unreadable past a handful of classes.
CM_ANNOTATE_MAX_CLASSES = 12
CM_COLORMAP = "magma"


def summary_path_for(csv_path):
    """results/<run>/epoch_metrics.csv -> results/<run>.json.

    Built from the directory name rather than Path.with_suffix, which would
    truncate at the first dot in a run name.
    """
    run_dir = Path(csv_path).parent
    return run_dir.parent / f"{run_dir.name}.json"


def load_confusion(csv_path, key):
    """(matrix, classes) for one run, or (None, None) if not written yet.

    The summary JSON only appears when a run finishes its final evaluation, so
    a run that is still training -- or was preempted before the end -- has an
    epoch_metrics.csv but no matrix. That is a normal mid-sweep state, not an
    error.
    """
    path = summary_path_for(csv_path)
    if not path.exists():
        return None, None
    payload = json.loads(path.read_text())
    cm = (payload.get(key) or {}).get("confusion_matrix")
    if cm is None:
        return None, None
    return np.asarray(cm, dtype=float), payload.get("classes")


def row_normalize(cm):
    """Rows -> per-class recall. Classes with no test support stay all-zero
    rather than dividing by zero, and are excluded from the mean diagonal."""
    totals = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, totals, out=np.zeros_like(cm), where=totals > 0)


def confusion_grid(pairs, group_name, key, granularity):
    loaded = []
    pending = []
    for label, path in available_runs(pairs, group_name):
        cm, classes = load_confusion(path, key)
        (loaded if cm is not None else pending).append(
            (label, cm, classes) if cm is not None else label
        )
    if pending:
        print(f"  [{group_name}] no {granularity}-level matrix yet (run unfinished): "
              f"{', '.join(pending)}")
    if not loaded:
        print(f"  [{group_name}] nothing to plot at {granularity} level.")
        return

    n_classes = len(loaded[0][2])
    ncols = min(3, len(loaded))
    nrows = -(-len(loaded) // ncols)  # ceil
    panel = max(3.0, 0.42 * n_classes)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(panel * ncols + 1.4, panel * nrows), squeeze=False
    )
    flat = axes.ravel()

    image = None
    for ax, (label, cm, classes) in zip(flat, loaded):
        norm = row_normalize(cm)
        supported = cm.sum(axis=1) > 0
        bacc = float(np.diag(norm)[supported].mean()) if supported.any() else float("nan")
        image = ax.imshow(norm, cmap=CM_COLORMAP, vmin=0.0, vmax=1.0)
        ax.set_title(f"{label}\nbalanced acc {bacc:.3f}", fontsize=9)
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=90, fontsize=7)
        ax.set_yticklabels(classes, fontsize=7)
        ax.set_xlabel("predicted", fontsize=8)
        ax.set_ylabel("true", fontsize=8)
        if len(classes) <= CM_ANNOTATE_MAX_CLASSES:
            for i in range(len(classes)):
                for j in range(len(classes)):
                    v = norm[i, j]
                    if v > 0.005:
                        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                                color="white" if v < 0.6 else "black")

    for ax in flat[len(loaded):]:
        ax.axis("off")

    fig.suptitle(f"{group_name}: {granularity}-level confusion (row-normalized)", fontsize=11)
    fig.colorbar(image, ax=axes, fraction=0.02, pad=0.02, label="fraction of true class")
    plt.show()


for granularity, key in CM_GRANULARITIES:
    for group_name, pairs in COMPARISON_GROUPS:
        confusion_grid(pairs, group_name, key, granularity)